## _Segmenting Stage_

- _Testing the **Segmenting** LightningModule_

In [1]:
import glob, os, sys, yaml

In [2]:
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
import pprint
pp = pprint.PrettyPrinter(indent=2)
import seaborn as sns
import trackml.dataset

In [4]:
import torch
from torch_geometric.data import Data
import itertools

In [5]:
# Find current device.
device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
# Set environemnt variable.
# os.environ['EXATRKX_DATA'] = os.path.abspath(os.curdir)

In [7]:
os.environ['EXATRKX_DATA'] = '/shared/adeel/data_sets/ctd2022'

### _3.1 - Configurtion_

In [8]:
# load processing config file (trusted source)
config_path = 'LightningModules/Segmenting/configs/segment_quickstart.yaml'
config_file = os.path.join(os.curdir, config_path)
with open(config_file) as f:
    try:
        config = yaml.load(f, Loader=yaml.FullLoader) # equiv: yaml.full_load(f)
    except yaml.YAMLError as e:
        print(e)

In [9]:
pp.pprint(config)

{ 'datatype_names': [['train', 'val', 'test']],
  'edge_cut': 0.5,
  'input_dir': '${EXATRKX_DATA}/run_quick/gnn_processed',
  'n_files': 10,
  'n_tasks': 1,
  'n_workers': 1,
  'output_dir': '${EXATRKX_DATA}/run_quick/seg_processed'}


### _3.2 - Input Data_

In [10]:
# fetch all files
inputdir = '../ctd2022/run_all/gnn_evaluation/test'
gnn_files = sorted(glob.glob(os.path.join(inputdir, "*")))

In [11]:
event_idx = 1

In [12]:
gnn_data = torch.load(gnn_files[event_idx], map_location=device)
print("Length of Data: {}".format(len(gnn_data)))

Length of Data: 11


In [13]:
gnn_data

Data(x=[171, 3], pid=[171], layers=[171], event_file='/global/cscratch1/sd/aakram/train_all/event0000095001', hid=[171], pt=[171], modulewise_true_edges=[2, 160], layerwise_true_edges=[2, 172], edge_index=[2, 808], y_pid=[808], scores=[1616])

In [14]:
gnn_data.x[:10]

tensor([[ 0.1663,  0.0097,  0.0036],
        [ 0.1663, -0.6570,  0.0022],
        [ 0.1669,  0.6956,  0.0025],
        [ 0.1669, -0.3623,  0.0018],
        [ 0.1669,  0.6956,  0.0020],
        [ 0.1669, -0.6377,  0.0019],
        [ 0.1681,  0.2853,  0.0032],
        [ 0.1681, -0.7147,  0.0033],
        [ 0.1681,  0.6187,  0.0019],
        [ 0.1723, -0.9150,  0.0009]], grad_fn=<SliceBackward0>)

In [15]:
gnn_data.event_file

'/global/cscratch1/sd/aakram/train_all/event0000095001'

In [16]:
type(gnn_data.y_pid)

torch.Tensor

### _3.3 - Segmentation_

- Segmentation is the Connected-Components Labeling (CCL) of Graphs base on Edge Score.

In [17]:
from LightningModules.Segmenting.utils.segmentation_utils import label_graph

In [18]:
graph = torch.load(gnn_files[0], map_location="cpu")

In [19]:
graph

Data(x=[148, 3], pid=[148], layers=[148], event_file='/global/cscratch1/sd/aakram/train_all/event0000095000', hid=[148], pt=[148], modulewise_true_edges=[2, 139], layerwise_true_edges=[2, 143], edge_index=[2, 961], y_pid=[961], scores=[1922])

In [23]:
edge_cut = 0.5

In [32]:
passing_edges = graph.edge_index[:, graph.scores[:graph.edge_index.shape[1]] > edge_cut]

In [34]:
# only few edges left
passing_edges.shape

torch.Size([2, 145])

In [37]:
import scipy.sparse.csgraph as scigraph
import scipy.sparse as sp

In [38]:
# get connected components
sparse_edges = sp.coo_matrix(
    (np.ones(passing_edges.shape[1]), passing_edges.cpu().numpy()),
    shape=(len(graph.x), len(graph.x)),
)

In [40]:
sparse_edges.shape

(148, 148)

In [42]:
scigraph.connected_components(sparse_edges)[0]

8

In [43]:
scigraph.connected_components(sparse_edges)[1]

array([0, 1, 2, 3, 4, 5, 1, 6, 2, 0, 7, 7, 3, 6, 4, 5, 2, 0, 1, 7, 7, 3,
       6, 4, 5, 2, 1, 0, 3, 7, 7, 4, 2, 6, 1, 0, 5, 7, 3, 6, 7, 7, 4, 5,
       2, 0, 1, 7, 3, 5, 4, 6, 7, 1, 0, 2, 3, 7, 7, 4, 0, 1, 5, 6, 2, 7,
       3, 7, 7, 5, 6, 4, 6, 7, 7, 0, 2, 1, 3, 2, 0, 5, 6, 4, 1, 0, 4, 6,
       2, 3, 1, 5, 0, 2, 7, 3, 6, 4, 1, 5, 7, 0, 2, 7, 3, 1, 6, 4, 5, 7,
       0, 2, 6, 3, 1, 3, 4, 7, 5, 7, 0, 2, 4, 6, 3, 1, 5, 7, 0, 7, 2, 4,
       6, 3, 5, 1, 7, 0, 2, 6, 7, 1, 5, 4, 3, 7, 0, 2], dtype=int32)

In [44]:
connected_components = scigraph.connected_components(sparse_edges)[1]

In [45]:
# attach labels to data
graph.labels = connected_components

In [47]:
graph.labels

array([0, 1, 2, 3, 4, 5, 1, 6, 2, 0, 7, 7, 3, 6, 4, 5, 2, 0, 1, 7, 7, 3,
       6, 4, 5, 2, 1, 0, 3, 7, 7, 4, 2, 6, 1, 0, 5, 7, 3, 6, 7, 7, 4, 5,
       2, 0, 1, 7, 3, 5, 4, 6, 7, 1, 0, 2, 3, 7, 7, 4, 0, 1, 5, 6, 2, 7,
       3, 7, 7, 5, 6, 4, 6, 7, 7, 0, 2, 1, 3, 2, 0, 5, 6, 4, 1, 0, 4, 6,
       2, 3, 1, 5, 0, 2, 7, 3, 6, 4, 1, 5, 7, 0, 2, 7, 3, 1, 6, 4, 5, 7,
       0, 2, 6, 3, 1, 3, 4, 7, 5, 7, 0, 2, 4, 6, 3, 1, 5, 7, 0, 7, 2, 4,
       6, 3, 5, 1, 7, 0, 2, 6, 7, 1, 5, 4, 3, 7, 0, 2], dtype=int32)

In [49]:
np.where(graph.labels==0)

(array([  0,   9,  17,  27,  35,  45,  54,  60,  75,  80,  85,  92, 101,
        110, 120, 128, 137, 146]),)